In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# Wikipedia Retriever
from langchain_community.retrievers import WikipediaRetriever
import wikipedia

retriever = WikipediaRetriever(top_k_results=2,lang="en",wiki_client=wikipedia)

query = """
Which Christopher Nolan movie is about a thief who enters people's dreams
to steal information and manipulate ideas?
"""

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print("=" * 60)
    print(f"🎬 Document {i}")
    print("=" * 60)

    print(doc.page_content.strip())

    print("=" * 60)
    print()

🎬 Document 1
Inception is a 2010  science fiction action film written and directed by Christopher Nolan, who also produced it with Emma Thomas, his wife. The film stars Leonardo DiCaprio as a professional thief who steals information by infiltrating the subconscious of his targets. He is offered a chance to have his criminal history erased as payment for the implantation of another person's idea into a target's subconscious. The ensemble cast includes Ken Watanabe, Joseph Gordon-Levitt, Marion Cotillard, Elliot Page, Tom Hardy, Cillian Murphy, Tom Berenger, Dileep Rao, and Michael Caine.
After the completion of Insomnia in 2002, Nolan presented to Warner Bros. a written 80-page treatment for a horror film envisioning "dream stealers," based on lucid dreaming. Deciding he needed more experience before tackling a production of this magnitude and complexity, Nolan shelved the project and instead worked on Batman Begins (2005), The Prestige (2006), and The Dark Knight (2008). The treatment

In [ ]:
# Vector Store
from langchain_core.documents import Document
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings


documents = [
    Document(
        page_content="""
        Inception is a science fiction thriller directed by Christopher Nolan.
        The story follows Dom Cobb, a skilled thief who enters people's dreams
        to steal valuable information. Cobb is offered a chance to erase his
        criminal history by planting an idea inside someone's mind. The movie
        explores dreams, memory, reality, and the consequences of manipulating
        the human mind.
        """,
        metadata={
            "title": "Inception",
            "year": 2010,
            "genre": "Science Fiction",
            "director": "Christopher Nolan"
        }
    ),

    Document(
        page_content="""
        The Dark Knight is a superhero crime thriller directed by Christopher
        Nolan. Batman faces the Joker, a dangerous criminal who creates chaos
        throughout Gotham City. The movie focuses on the conflict between
        Batman's sense of justice and the Joker's belief that society can be
        pushed into violence and disorder.
        """,
        metadata={
            "title": "The Dark Knight",
            "year": 2008,
            "genre": "Superhero",
            "director": "Christopher Nolan"
        }
    ),

    Document(
        page_content="""
        Interstellar is a science fiction movie directed by Christopher Nolan.
        A group of astronauts travels through a wormhole in search of a new
        home for humanity. Cooper, a former pilot, leaves his family behind to
        explore distant planets. The movie deals with space exploration,
        gravity, time, love, and humanity's survival.
        """,
        metadata={
            "title": "Interstellar",
            "year": 2014,
            "genre": "Science Fiction",
            "director": "Christopher Nolan"
        }
    ),

    Document(
        page_content="""
        The Matrix is a science fiction action movie directed by the
        Wachowskis. Neo discovers that the world he knows is actually a
        simulated reality created by machines. With the help of Morpheus and
        Trinity, he learns the truth and joins a rebellion against the
        machines. The movie explores artificial intelligence, reality,
        freedom, and human consciousness.
        """,
        metadata={
            "title": "The Matrix",
            "year": 1999,
            "genre": "Science Fiction",
            "director": "The Wachowskis"
        }
    ),

    Document(
        page_content="""
        Forrest Gump is a drama movie directed by Robert Zemeckis. The story
        follows Forrest, a kind and simple man who experiences many important
        events in American history. Despite his limited understanding of the
        world, Forrest succeeds in running, playing football, serving in the
        military, and starting a successful shrimp business. The movie explores
        friendship, love, family, and perseverance.
        """,
        metadata={
            "title": "Forrest Gump",
            "year": 1994,
            "genre": "Drama",
            "director": "Robert Zemeckis"
        }
    )
]

embedding = MistralAIEmbeddings(model="mistral-embed")

vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embedding
)

retriever = vector_store.as_retriever(search_kwargs={"k":2})
query = "A movie about entering people's dreams"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print("=" * 60)
    print(f"🎬 Document {i} - {doc.metadata['title']}")
    print("=" * 60)

    print(doc.page_content.strip())

    print("=" * 60)
    print()


🎬 Document 1 - Inception
Inception is a science fiction thriller directed by Christopher Nolan.
        The story follows Dom Cobb, a skilled thief who enters people's dreams
        to steal valuable information. Cobb is offered a chance to erase his
        criminal history by planting an idea inside someone's mind. The movie
        explores dreams, memory, reality, and the consequences of manipulating
        the human mind.

🎬 Document 2 - The Matrix
The Matrix is a science fiction action movie directed by the
        Wachowskis. Neo discovers that the world he knows is actually a
        simulated reality created by machines. With the help of Morpheus and
        Trinity, he learns the truth and joins a rebellion against the
        machines. The movie explores artificial intelligence, reality,
        freedom, and human consciousness.



In [19]:
# MMR (Maximum Marginal Relevance) - More Diverse Result
from langchain_core.documents import Document
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings

documents = [
    Document(page_content="Global warming is causing average temperatures to rise and increasing the frequency of extreme heat waves."),
    Document(page_content="Climate change is melting glaciers and polar ice, contributing to rising sea levels around the world."),
    Document(page_content="Deforestation contributes to global warming because fewer trees are available to absorb carbon dioxide from the atmosphere."),
    Document(page_content="Burning coal, oil, and natural gas releases greenhouse gases that trap heat and contribute to global warming."),
    Document(page_content="Global warming is affecting agriculture by changing rainfall patterns, increasing droughts, and reducing crop productivity."),
]
embedding = MistralAIEmbeddings(model="mistral-embed")

vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embedding
)

retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={"k": 3, "lambda_mult": 0.2}
)

# retriever = vector_store.as_retriever(
#     search_kwargs={"k": 3}
# )

query = "How does global warming affect the environment?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"🎬 Document {i}")
    print(doc.page_content.strip())
    print()


🎬 Document 1
Global warming is causing average temperatures to rise and increasing the frequency of extreme heat waves.

🎬 Document 2
Deforestation contributes to global warming because fewer trees are available to absorb carbon dioxide from the atmosphere.

🎬 Document 3
Burning coal, oil, and natural gas releases greenhouse gases that trap heat and contribute to global warming.



In [23]:
# MultiQuery Retrieval
# Query → LLM → Generate Multiple Queries → Retrieve Documents → Merge Results → Remove Duplicates → Return Results

from langchain_core.documents import Document
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

documents = [
    Document(page_content="Drinking enough water helps keep the body hydrated and healthy."),
    Document(page_content="Walking every day is a simple way to improve fitness and heart health."),
    Document(page_content="Eating fresh fruits and vegetables provides important vitamins and minerals."),
    Document(page_content="Getting seven to eight hours of sleep helps the body and mind recover."),
    Document(page_content="Regular exercise can improve mood and reduce stress."),
    Document(page_content="Taking short breaks while working can reduce tiredness and improve focus."),
    Document(page_content="Spending time outdoors and getting sunlight can support overall wellbeing."),
    Document(page_content="The fitness center has new machines for weight training and cardio workouts."),
    Document(page_content="The hotel provides healthy breakfast options, a swimming pool, and a fitness room."),
    Document(page_content="The health insurance company offers different plans for medical expenses."),
]

query = "How can I stay healthy?"

embedding = MistralAIEmbeddings(model="mistral-embed")

vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embedding
)

similarity_search = vector_store.as_retriever(search_kwargs={"k": 5})
multiquery_search = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
    llm=ChatMistralAI(model="mistral-medium-3-5")
)

similarity_result = similarity_search.invoke(query)
multiquery_result = multiquery_search.invoke(query)

for i, doc in enumerate(similarity_result, 1):
    print(f"🎬 Document {i}")
    print(doc.page_content.strip())
    print()

print("=" * 60)

for i, doc in enumerate(multiquery_result, 1):
    print(f"🎬 Document {i}")
    print(doc.page_content.strip())
    print()

🎬 Document 1
Drinking enough water helps keep the body hydrated and healthy.

🎬 Document 2
Eating fresh fruits and vegetables provides important vitamins and minerals.

🎬 Document 3
Spending time outdoors and getting sunlight can support overall wellbeing.

🎬 Document 4
The hotel provides healthy breakfast options, a swimming pool, and a fitness room.

🎬 Document 5
Walking every day is a simple way to improve fitness and heart health.

🎬 Document 1
Drinking enough water helps keep the body hydrated and healthy.

🎬 Document 2
Eating fresh fruits and vegetables provides important vitamins and minerals.

🎬 Document 3
Regular exercise can improve mood and reduce stress.

🎬 Document 4
Walking every day is a simple way to improve fitness and heart health.

🎬 Document 5
Spending time outdoors and getting sunlight can support overall wellbeing.

🎬 Document 6
Taking short breaks while working can reduce tiredness and improve focus.

🎬 Document 7
Getting seven to eight hours of sleep helps the b

In [26]:
# Contextual  Compression Retrieval 
from langchain_core.documents import Document
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings
from langchain_mistralai import ChatMistralAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

documents = [
    Document(page_content="""
    Photosynthesis is the process by which green plants make their own food.
    Plants use sunlight, water, and carbon dioxide to produce glucose and oxygen.
    The process mainly takes place in the leaves. Chlorophyll captures sunlight
    and provides the energy needed for photosynthesis. Plants also need healthy
    roots to absorb water from the soil.
    """),

    Document(page_content="""
    Photosynthesis plays an important role in the environment because it removes
    carbon dioxide from the atmosphere and releases oxygen. Forests contain
    millions of plants that perform photosynthesis every day. Trees also provide
    shelter for animals and help prevent soil erosion. Healthy forests are
    important for maintaining biodiversity.
    """),

    Document(page_content="""
    Photosynthesis depends heavily on sunlight and temperature. When there is
    not enough light, the rate of photosynthesis decreases. Very high
    temperatures can also reduce photosynthesis because plant enzymes may not
    work properly. Temperature also affects plant growth, water loss, and
    agricultural production.
    """),

    Document(page_content="""
    Photosynthesis requires carbon dioxide, which plants obtain from the air.
    When carbon dioxide enters a leaf through tiny openings called stomata, it
    becomes available for photosynthesis. Stomata also control the movement of
    water vapor out of the plant. Different plants have different adaptations
    for controlling water loss.
    """),

    Document(page_content="""
    Photosynthesis produces glucose, which plants can use as a source of energy.
    Plants can also convert glucose into starch and store it for later use.
    Starch is commonly stored in roots, seeds, and fruits. These stored nutrients
    are important for plant growth and are also an important source of food for
    humans and animals.
    """),
]

query = "What does a plant need for photosynthesis?"
# query = "What happens when there is not enough sunlight for photosynthesis?"

embedding = MistralAIEmbeddings(model="mistral-embed")

vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embedding
)

base_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

llm = ChatMistralAI(model="mistral-medium-3-5")
compressor = LLMChainExtractor.from_llm(llm)

compression_retrieval = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

docs = compression_retrieval.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"🎬 Document {i}")
    print(doc.page_content.strip())
    print()


🎬 Document 1
Extracted relevant parts:
Plants use sunlight, water, and carbon dioxide to produce glucose and oxygen.

🎬 Document 2
Extracted relevant parts:
Photosynthesis requires carbon dioxide, which plants obtain from the air.

🎬 Document 3
Extracted relevant parts:
Photosynthesis depends heavily on sunlight and temperature.

